# 06 — Proponentes por Proceso SECOP II

**Fuente**: `proponentesPorProceso` (Dataset ID: `hgi6-6wh3`)  
**Granularidad**: 1 fila = 1 proponente que presentó oferta en 1 proceso  
**Columnas disponibles**: 9  

**Objetivo**: Descargar los proponentes que participaron en procesos de contratación de Obra Pública,
para luego extraer su información financiera del RUP mediante el scraper automático.

**Estrategia de descarga**: El dataset completo tiene ~2.25M registros de todos los tipos de contrato.
Filtramos desde la API usando los `id_del_proceso` de procesos de Obra que ya tenemos en parquet.

**Llave de integración**: `id_procedimiento` (proponentes) = `id_del_proceso` (procesosDeContratacion)

**Por qué este notebook es necesario.** El dataset de `procesosDeContratacion` (notebook 02) ya trae un campo agregado de "número de oferentes" que calcula el propio SECOP, pero ese conteo agregado no siempre coincide con el detalle real de quién se presentó a competir, y no permite identificar *quiénes* fueron esos oferentes (su NIT, su razón social). Para verificar ese agregado y, sobre todo, para poder cruzar cada proponente contra su información financiera en el RUP (Registro Único de Proponentes), es necesario bajar un dataset distinto y más granular: uno donde cada fila es un proponente que efectivamente radicó oferta en un proceso puntual.

| Campo API | Tipo | Descripción |
|---|---|---|
| id_procedimiento | Texto | Identificador del procedimiento (CO1.REQ.XXX) |
| fecha_publicaci_n | Timestamp | Fecha de publicación del proceso |
| nombre_procedimiento | Texto | Nombre del proceso |
| nit_entidad | Número | NIT de la entidad contratante |
| codigo_entidad | Número | Código de la entidad contratante |
| entidad_compradora | Texto | Nombre de la entidad compradora |
| proveedor | Texto | Nombre del proveedor/proponente |
| nit_proveedor | Texto | NIT del proveedor |
| codigo_proveedor | Número | Código SECOP II del proveedor |

**Hacia dónde va este resultado.** Esta descarga alimenta dos productos que se usan más adelante en el trabajo: (1) un conteo de proponentes reales por proceso, que se cruza en el EDA extendido con el identificador de portafolio de cada contrato para construir la variable de "competencia real" usada para poner a prueba la hipótesis H0 (¿la competencia entre empresas influye en la duración del contrato?); y (2) una lista depurada de NITs de proponentes que sirve de insumo al scraper automático del RUP, para enriquecer cada empresa con indicadores financieros (liquidez, endeudamiento, tamaño, etc.).

---

## 1. Configuración

Se replica exactamente el patrón de conexión usado en los notebooks anteriores de esta serie (03, 04, 05): mismo `APP_TOKEN`, mismo dominio `www.datos.gov.co` y la misma función `fetch_with_retry` con reintentos y espera creciente (*backoff* de 10, 20, 30... segundos). Mantener este patrón idéntico entre notebooks es una decisión deliberada: como cada notebook descarga un dataset distinto pero con el mismo mecanismo de fallas típico de la API de Socrata (timeouts y errores intermitentes en descargas grandes por lotes), reutilizar la misma función de reintentos reduce el riesgo de errores de transcripción y facilita mantener el código. Lo único que cambia respecto a los notebooks previos es el `DATASET_ID` (`hgi6-6wh3`, correspondiente a `proponentesPorProceso`). La salida `OK` confirma que el cliente Socrata se instanció correctamente y que el entorno (pandas, numpy, matplotlib, seaborn) quedó listo antes de iniciar cualquier descarga.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sodapy import Socrata
import warnings
import time

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

# Constantes de conexión
APP_TOKEN  = 'nrnMDVHrooMY3wZIGrYfPsAq3'
DOMAIN     = 'www.datos.gov.co'
DATASET_ID = 'hgi6-6wh3'  # proponentesPorProceso
DATA_DIR   = '../data/'

client = Socrata(DOMAIN, APP_TOKEN, timeout=120)

def fetch_with_retry(client, dataset_id, max_retries=5, **kwargs):
    """Consulta Socrata con reintentos y backoff."""
    for attempt in range(max_retries):
        try:
            return client.get(dataset_id, **kwargs)
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 10 * (attempt + 1)
                print(f'    Error: {type(e).__name__}. Reintentando en {wait}s... (intento {attempt+2}/{max_retries})')
                time.sleep(wait)
            else:
                raise

print("OK")

OK


## 2. Cargar IDs de procesos de Obra

Antes de descargar un solo registro de proponentes, se carga el parquet `procesos_contratacion_obra.parquet` — el resultado ya filtrado por Obra Pública que se generó en el notebook 02 — y se extraen sus identificadores de proceso (`id_del_proceso`, formato `CO1.REQ.XXX`). Esta es la misma decisión metodológica aplicada en el notebook 03: en lugar de descargar el dataset completo de `proponentesPorProceso` (que según su documentación contiene del orden de 2.25 millones de registros de *todos* los tipos de contratación: obra, servicios, compras, consultoría, etc.), se usa la lista de procesos de Obra ya depurada como filtro, evitando así traer a memoria millones de registros irrelevantes para esta investigación.

El resultado real confirma la magnitud del filtro: el parquet trae **138,112 registros**, pero solo **112,891 valores únicos** de `id_del_proceso` — es decir, en promedio cada proceso aparece más de una vez en `procesosDeContratacion` (por ejemplo, cuando un mismo proceso tiene varios ítems o líneas de contrato asociadas). Por eso el paso `dropna().unique().tolist()` es indispensable: sin la deduplicación, se estarían generando lotes de descarga con IDs repetidos, desperdiciando llamadas a la API. El formato de ejemplo (`CO1.REQ.848621`) es el que se usará literalmente dentro de las cláusulas `WHERE ... IN (...)` de la sección 4.

In [2]:
df_procesos = pd.read_parquet(DATA_DIR + 'procesos_contratacion_obra.parquet')
obra_ids = df_procesos['id_del_proceso'].dropna().unique().tolist()

print(f"Registros en parquet:       {len(df_procesos):,}")
print(f"id_del_proceso únicos:      {len(obra_ids):,}")
print(f"Formato ejemplo:            {obra_ids[0]}")

Registros en parquet:       138,112
id_del_proceso únicos:      112,891
Formato ejemplo:            CO1.REQ.848621


## 3. Verificar nombres de campos en la API

Antes de lanzar una descarga de cientos de lotes, se trae un único registro de muestra (`limit=1`) para confirmar cómo llegan realmente los nombres de columna desde la API — Socrata normaliza los nombres de campo originales (por ejemplo, quita tildes y caracteres especiales, y trunca o renombra según sus propias reglas internas), por lo que no es seguro asumir que el nombre de columna coincide exactamente con el que aparece en la documentación del dataset. Verificarlo con una sola llamada barata evita fallos de tipo `KeyError` a mitad de una descarga de 753 lotes.

El resultado también deja ver, de forma incidental, un problema de codificación de caracteres propio de esta fuente: valores como `HuMEDOS` (en vez de "HÚMEDOS") o `BATALLoN` (en vez de "BATALLÓN") muestran que las vocales acentuadas y la Ñ llegan mal transcritas desde el origen. Es una limitación de calidad de datos que se documenta aquí porque es visible directamente en la salida, aunque su corrección (si se necesitara para texto libre) quedaría fuera del alcance de este notebook, cuyo interés está en los campos identificadores (`id_procedimiento`, `nit_proveedor`) y no en los campos de texto libre.

In [3]:
sample = fetch_with_retry(client, DATASET_ID, limit=1)
api_columns = list(sample[0].keys())
print("Campos reales en la API:")
for col in api_columns:
    print(f"  - {col}: {sample[0][col][:80] if isinstance(sample[0][col], str) else sample[0][col]}")

Campos reales en la API:
  - id_procedimiento: CO1.REQ.10159505
  - fecha_publicaci_n: 2026-02-11T00:00:00.000
  - nombre_procedimiento: SUMINISTRO DE PAÑALES DESECHABLES Y PAÑITOS HuMEDOS ESM-BAS06
  - nit_entidad: 901541245
  - codigo_entidad: 702962226
  - entidad_compradora: ESM BATALLoN ASPC NO. 6
  - proveedor: INTERCOMERCIAL MEDICA
  - nit_proveedor: 830501223
  - codigo_proveedor: 700150014


## 4. Descarga por lotes

Se descarga `proponentesPorProceso` filtrando con una cláusula `WHERE id_procedimiento IN (...)` construida con lotes de 150 IDs a la vez — el mismo patrón usado en el notebook 03 para las adiciones. El tamaño de lote no es arbitrario: una cláusula `IN` con demasiados valores generaría una URL de consulta excesivamente larga (riesgo de error HTTP 414 o de rechazo por parte de Socrata), mientras que lotes muy pequeños multiplicarían innecesariamente el número de llamadas a la API. 150 IDs por lote es el punto de equilibrio ya validado en notebooks anteriores de esta serie.

Con **112,891 IDs** de procesos de Obra a razón de 150 por lote, se generan **753 lotes**. Dentro de cada lote además se pagina con `offset` en pasos de 50,000 registros (el límite máximo por consulta de Socrata) — una salvaguarda necesaria porque, aunque es poco probable que 150 procesos concentren más de 50,000 proponentes combinados, algunos procesos pueden tener cientos de oferentes (como se confirma más adelante, hasta 255 en un solo proceso), y sin esta paginación interna se truncaría silenciosamente el resultado. El progreso se imprime cada 50 lotes para poder monitorear la descarga sin saturar la salida. Al completarse los 753 lotes, se acumulan **276,770 registros** de proponentes — el insumo crudo sobre el que se construye todo el resto del notebook.

In [4]:
chunk_size = 150
all_records = []
total_chunks = (len(obra_ids) + chunk_size - 1) // chunk_size

print(f"IDs de procesos de Obra:  {len(obra_ids):,}")
print(f"Lotes de {chunk_size} IDs:          {total_chunks}")
print()

for i in range(0, len(obra_ids), chunk_size):
    chunk = obra_ids[i:i+chunk_size]
    chunk_num = i // chunk_size + 1

    # Construir cláusula WHERE IN
    ids_str = ", ".join([f"'{pid}'" for pid in chunk])
    where_clause = f"id_procedimiento IN ({ids_str})"

    # Descargar todos los proponentes para este lote (con paginación)
    offset = 0
    while True:
        batch = fetch_with_retry(client, DATASET_ID,
                                 where=where_clause,
                                 limit=50000,
                                 offset=offset)
        if not batch:
            break
        all_records.extend(batch)
        if len(batch) < 50000:
            break
        offset += 50000

    if chunk_num % 50 == 0 or chunk_num == total_chunks:
        print(f"  Lote {chunk_num}/{total_chunks} — Registros acumulados: {len(all_records):,}")

print(f"\n✓ Descarga completa: {len(all_records):,} registros")

IDs de procesos de Obra:  112,891
Lotes de 150 IDs:          753



  Lote 50/753 — Registros acumulados: 28,311


  Lote 100/753 — Registros acumulados: 49,046


  Lote 150/753 — Registros acumulados: 68,958


  Lote 200/753 — Registros acumulados: 88,334


...

  Lote 600/753 — Registros acumulados: 226,037


  Lote 650/753 — Registros acumulados: 242,518


  Lote 700/753 — Registros acumulados: 258,438


  Lote 750/753 — Registros acumulados: 275,628


  Lote 753/753 — Registros acumulados: 276,770

✓ Descarga completa: 276,770 registros


## Consolidación en DataFrame

Los `276,770` registros descargados llegan como una lista de diccionarios (`all_records`), formato nativo de las respuestas de Socrata. Antes de poder perfilar o cruzar esta información, se convierte a un `DataFrame` de pandas con `pd.DataFrame.from_records(...)`. El `shape` resultante, `(276770, 9)`, confirma dos cosas a la vez: que no se perdió ni un registro en la conversión (coincide exactamente con el conteo acumulado del paso anterior) y que las 9 columnas documentadas en la cabecera del notebook llegaron completas. A partir de aquí todo el trabajo se hace sobre este DataFrame `df`, que es la tabla de trabajo principal del notebook.

In [5]:
df = pd.DataFrame.from_records(all_records)
print(f"Shape: {df.shape}")
print(f"Columnas: {list(df.columns)}")
df.head()

Shape: (276770, 9)
Columnas: ['id_procedimiento', 'fecha_publicaci_n', 'nombre_procedimiento', 'nit_entidad', 'codigo_entidad', 'entidad_compradora', 'proveedor', 'nit_proveedor', 'codigo_proveedor']


,id_procedimiento,fecha_publicaci_n,nombre_procedimiento,nit_entidad,codigo_entidad,entidad_compradora,proveedor,nit_proveedor,codigo_proveedor
0,CO1.REQ.10064540,2026-01-30T00:00:00.000,CONTRATAR LA TERMINACIoN MEJORAMIENTO REMODELACIoN ADECUACIoN MODIFICACIoN MANTENIMIENTO PREVENT...,800141686,701746083,ARMADA NACIONAL - BASE NAVAL NO. 3 ARC LEGUIZAMO.*,CONSORCIO SAIFOR,1102868937,734498561
1,CO1.REQ.10064540,2026-01-30T00:00:00.000,CONTRATAR LA TERMINACIoN MEJORAMIENTO REMODELACIoN ADECUACIoN MODIFICACIoN MANTENIMIENTO PREVENT...,800141686,701746083,ARMADA NACIONAL - BASE NAVAL NO. 3 ARC LEGUIZAMO.*,CONSORCIO VENTURA & CO.,No Definido,734861826
2,CO1.REQ.10064540,2026-01-30T00:00:00.000,CONTRATAR LA TERMINACIoN MEJORAMIENTO REMODELACIoN ADECUACIoN MODIFICACIoN MANTENIMIENTO PREVENT...,800141686,701746083,ARMADA NACIONAL - BASE NAVAL NO. 3 ARC LEGUIZAMO.*,CONSTRUCTORA AHA S.A.S,830010208,701699522
3,CO1.REQ.10064540,2026-01-30T00:00:00.000,CONTRATAR LA TERMINACIoN MEJORAMIENTO REMODELACIoN ADECUACIoN MODIFICACIoN MANTENIMIENTO PREVENT...,800141686,701746083,ARMADA NACIONAL - BASE NAVAL NO. 3 ARC LEGUIZAMO.*,SAMAR CONSTRUCTORES Y CONSULTORES SAS,900088297,700844095
4,CO1.REQ.10064540,2026-01-30T00:00:00.000,CONTRATAR LA TERMINACIoN MEJORAMIENTO REMODELACIoN ADECUACIoN MODIFICACIoN MANTENIMIENTO PREVENT...,800141686,701746083,ARMADA NACIONAL - BASE NAVAL NO. 3 ARC LEGUIZAMO.*,MARCELINO,No Definido,734906878


## 5. Perfilado rápido

Antes de usar esta tabla para cualquier cálculo aguas abajo, se hace un perfilado básico: cuántos registros, cuántos procesos y proponentes distintos hay, cuántos nulos y valores centinela (`'No Definido'`) contiene, y cómo se distribuye el número de proponentes por proceso. Este paso es el que determina, en la práctica, qué tan confiable es la variable de "competencia real" que se construirá después.

Los resultados muestran que los **276,770 registros** corresponden a **33,171 procesos únicos** y **23,759 proponentes (NIT) únicos**, repartidos entre **2,264 entidades compradoras**. La calidad de los campos identificadores es alta —`fecha_publicaci_n` solo tiene 3 nulos—, pero el campo clave para identificar empresas, `nit_proveedor`, trae **93,103 valores `'No Definido'`** (cerca del 33.6% de los registros): es decir, uno de cada tres proponentes no tiene NIT capturado en el origen, lo que limita de entrada cuántas empresas se podrán enriquecer más adelante con datos del RUP.

La distribución de proponentes por proceso (`groupby('id_procedimiento')['nit_proveedor'].count()`) es la pieza más relevante para la hipótesis de competencia: la media es de **8.34** proponentes por proceso, pero la mediana es apenas **3**, y el 25% de los procesos tuvo **un solo proponente** (es decir, ni siquiera hubo competencia real, aunque formalmente el proceso se haya publicado como convocatoria abierta). En el otro extremo, el máximo llega a **255 proponentes** en un solo proceso, y la desviación estándar (**15.77**, casi el doble de la media) confirma que la distribución es fuertemente asimétrica hacia la derecha (unos pocos procesos muy competidos, la mayoría con poca o ninguna competencia real).

In [6]:
print("=== Perfilado básico ===\n")
print(f"Registros totales:          {len(df):,}")
print(f"Procesos únicos:            {df['id_procedimiento'].nunique():,}")
print(f"Proponentes únicos (NIT):   {df['nit_proveedor'].nunique():,}")
print(f"Entidades únicas:           {df['entidad_compradora'].nunique():,}")

print(f"\n--- Nulos por columna ---")
print(df.isnull().sum().to_string())

print(f"\n--- Valores 'No Definido' ---")
for col in df.columns:
    nd = (df[col] == 'No Definido').sum()
    if nd > 0:
        print(f"  {col}: {nd:,}")

print(f"\n--- Proponentes por proceso ---")
props_por_proc = df.groupby('id_procedimiento')['nit_proveedor'].count()
print(props_por_proc.describe())

=== Perfilado básico ===

Registros totales:          276,770
Procesos únicos:            33,171
Proponentes únicos (NIT):   23,759
Entidades únicas:           2,264

--- Nulos por columna ---
id_procedimiento        0
fecha_publicaci_n       3
nombre_procedimiento    0
nit_entidad             0
codigo_entidad          0
entidad_compradora      0
proveedor               0
nit_proveedor           0
codigo_proveedor        0

--- Valores 'No Definido' ---
  nit_proveedor: 93,103

--- Proponentes por proceso ---
count    33171.000000
mean         8.343734
std         15.770741
min          1.000000
25%          1.000000
50%          3.000000
75%          8.000000
max        255.000000
Name: nit_proveedor, dtype: float64


## 6. Cobertura vs procesosDeContratacion

¿Cuántos de los 112K procesos de Obra tienen al menos 1 proponente en este dataset?

Antes de dar por buena la variable de proponentes, hace falta responder una pregunta de cobertura: ¿de los 112,891 procesos de Obra que se identificaron en el notebook 02, cuántos aparecen realmente en este dataset de proponentes? Esto se resuelve con una intersección de conjuntos entre los IDs de proceso descargados aquí y los IDs de proceso del parquet original.

El resultado es una limitación metodológica importante que hay que documentar y tener presente en el análisis posterior: de los **112,891 procesos de Obra**, solo **33,171 (29.4%)** tienen al menos un proponente registrado en `proponentesPorProceso`; los otros **79,720 (70.6%)** no tienen ningún registro en esta fuente. Esto no significa necesariamente que esos procesos no tuvieron competencia — puede deberse a procesos que se tramitaron por modalidades o mecanismos que no quedan capturados en este dataset específico de SECOP (por ejemplo, contratación directa, mínima cuantía sin convocatoria pública formal, o vacíos históricos de reporte en la API). La consecuencia práctica es que la variable de "competencia real" que se construye a partir de esta fuente solo estará disponible, sin imputar, para menos de un tercio de los procesos de Obra — algo que debe manejarse explícitamente (no de forma implícita) al usarla en el EDA extendido y en las pruebas de hipótesis.

In [7]:
ids_con_proponentes = set(df['id_procedimiento'].unique())
ids_obra = set(obra_ids)

match = ids_con_proponentes & ids_obra
sin_proponentes = ids_obra - ids_con_proponentes

print(f"Procesos de Obra (parquet):     {len(ids_obra):,}")
print(f"Procesos con proponentes:       {len(match):,}  ({len(match)/len(ids_obra)*100:.1f}%)")
print(f"Procesos sin proponentes:       {len(sin_proponentes):,}  ({len(sin_proponentes)/len(ids_obra)*100:.1f}%)")

Procesos de Obra (parquet):     112,891


Procesos con proponentes:       33,171  (29.4%)
Procesos sin proponentes:       79,720  (70.6%)


## 7. NITs únicos de proponentes

Estos son los NITs que se pueden enviar al scraper RUP para obtener información financiera.

Con la tabla de proponentes ya perfilada, este paso extrae la lista de NITs que efectivamente sirven para identificar una empresa — es decir, se descartan los nulos, las cadenas vacías y el valor centinela `'No Definido'` que, como se vio en la sección 5, afecta a 93,103 registros. Esta lista es el puente hacia el scraper del RUP: cada NIT válido se convierte en una consulta individual contra el Registro Único de Proponentes para traer indicadores financieros de la empresa.

El resultado, **23,758 NITs únicos**, es casi idéntico al conteo de "proponentes únicos (NIT)" de la sección 5 (23,759) — la diferencia de exactamente 1 confirma que, de los valores únicos de `nit_proveedor`, el único que se estaba contando de más era el propio valor centinela `'No Definido'`, que ahora queda excluido. Vale la pena notar, mirando la muestra impresa, que no todos los NITs tienen un formato limpio (por ejemplo `'0001013594558'`, con más dígitos de los habituales en un NIT colombiano) — una irregularidad que se resuelve más adelante, en la sección 8, con una limpieza más estricta antes de enviar la lista al scraper.

In [8]:
nits_validos = df[
    (df['nit_proveedor'].notna()) &
    (df['nit_proveedor'] != 'No Definido') &
    (df['nit_proveedor'] != '')
]['nit_proveedor'].unique()

print(f"NITs de proponentes únicos:     {len(nits_validos):,}")
print(f"Primeros 10: {list(nits_validos[:10])}")

NITs de proponentes únicos:     23,758
Primeros 10: ['1102868937', '830010208', '900088297', '900392539', '901058985', '31655741', '0001013594558', '901346178', '901590857', '901455911']


## 8. Dataset unificado de proveedores para scraper RUP

Combinamos NITs de ambas fuentes en un solo dataset a nivel NIT:
- **procesos**: proponentes de `proponentesPorProceso` (Obra)
- **electronicos**: proveedores adjudicados de `contratosElectronicos` (Obra)

Columnas RUP vacías que llenará el scraper.

Hasta aquí solo se tiene la lista de NITs vista *desde el lado de los proponentes* (empresas que compitieron, hayan ganado o no). Pero el scraper del RUP necesita, idealmente, una única lista maestra de NITs para no consultar dos veces a la misma empresa ni dejar por fuera a proveedores que sí ganaron un contrato pero que, por alguna razón, no quedaron capturados como "proponentes" en la fuente de la sección 4. Por eso aquí se combinan dos orígenes distintos a nivel de NIT, tal como se listó arriba.

Combinar ambas fuentes antes de scrapear —en lugar de scrapear cada una por separado— es una decisión de eficiencia y de trazabilidad: evita duplicar consultas al RUP para el mismo NIT y, gracias a la columna `origen` (que puede tomar los valores `procesos`, `electronicos` o `ambos`), permite saber después si una empresa se identificó solo como proponente, solo como adjudicataria, o en ambos roles — información que en sí misma es relevante para caracterizar el mercado de contratistas de obra pública. Las columnas `rup_*` se agregan aquí vacías (como placeholders) porque las llenará un proceso externo posterior (el scraper automático del RUP), no este notebook; aquí solo se prepara la estructura y la lista de trabajo que ese scraper va a consumir.

In [ ]:
import re

# --- NITs de proponentes (procesos) ---
raw_prop = df['nit_proveedor'].copy()
raw_prop = raw_prop.apply(lambda x: x.split('-')[0] if isinstance(x, str) and re.match(r'^\d+-\d$', x) else x)
mask_prop = raw_prop.notna() & (raw_prop != 'No Definido') & raw_prop.str.match(r'^\d{6,10}$', na=False)

df_prop = df[mask_prop][['proveedor']].copy()
df_prop['nit'] = raw_prop[mask_prop]
nits_proc = df_prop.groupby('nit')['proveedor'].first().reset_index()
nits_proc['origen'] = 'procesos'
nits_proc.rename(columns={'proveedor': 'nombre_proveedor'}, inplace=True)
print(f"NITs limpios de procesos: {len(nits_proc):,}")

# --- NITs de contratos electrónicos ---
df_ce = pd.read_parquet(DATA_DIR + 'contratos_electronicos_obra.parquet')
raw_ce = df_ce['documento_proveedor'].copy()
raw_ce = raw_ce.apply(lambda x: x.split('-')[0] if isinstance(x, str) and re.match(r'^\d+-\d$', x) else x)
mask_ce = raw_ce.notna() & (raw_ce != 'No Definido') & raw_ce.str.match(r'^\d{6,10}$', na=False)

df_ce_nits = df_ce[mask_ce][['proveedor_adjudicado']].copy()
df_ce_nits['nit'] = raw_ce[mask_ce]
nits_elec = df_ce_nits.groupby('nit')['proveedor_adjudicado'].first().reset_index()
nits_elec['origen'] = 'electronicos'
nits_elec.rename(columns={'proveedor_adjudicado': 'nombre_proveedor'}, inplace=True)
print(f"NITs limpios de electrónicos: {len(nits_elec):,}")

# --- Unificar a nivel NIT ---
df_all = pd.concat([nits_proc, nits_elec], ignore_index=True)

origen_map = df_all.groupby('nit')['origen'].apply(
    lambda x: 'ambos' if set(x) == {'procesos', 'electronicos'} else x.iloc[0]
)
nombre_map = df_all.groupby('nit')['nombre_proveedor'].first()

df_unificado = pd.DataFrame({
    'nit': origen_map.index,
    'nombre_proveedor': nombre_map.values,
    'origen': origen_map.values,
})

print(f"\n=== Dataset unificado ===")
print(f"Total NITs únicos:    {len(df_unificado):,}")
print(f"  procesos:           {(df_unificado['origen'] == 'procesos').sum():,}")
print(f"  electronicos:       {(df_unificado['origen'] == 'electronicos').sum():,}")
print(f"  ambos:              {(df_unificado['origen'] == 'ambos').sum():,}")

## Normalización de NIT y columnas para el scraper

La celda anterior ya generó `df_unificado`; esta celda completa la tabla que efectivamente consumirá el scraper del RUP. Primero agrega, en blanco, las 17 columnas de información financiera y jurídica que se espera traer del RUP (estado de matrícula, tamaño de empresa, número de empleados, municipio, sanciones, multas, inhabilidades, y los indicadores financieros — activo total, patrimonio, ingresos, utilidad neta y los índices derivados de liquidez, endeudamiento, cobertura y rentabilidad). Dejarlas explícitamente definidas desde ahora, aunque vacías, fija el esquema final de la tabla y documenta qué información se espera poder incorporar más adelante.

Segundo, y más importante desde el punto de vista metodológico, se decide **qué NITs vale la pena enviarle al scraper** con la bandera `procesar`. Se excluyen dos categorías que no tienen sentido consultar en el RUP: los NITs compuestos solo por ceros (`'^0+$'`, un valor de relleno que no corresponde a ninguna empresa real) y los proponentes cuyo nombre indica que son **consorcios o uniones temporales** (detectados por patrones de texto como "CONSORCIO", "UNION TEMPORAL", "U.T."). Esta última exclusión responde a una particularidad del Registro Único de Proponentes: un consorcio no tiene RUP propio, porque es una figura contractual temporal formada por empresas que sí lo tienen individualmente — consultar el RUP con el NIT o nombre de un consorcio no devolvería información útil. La columna `procesado` (inicialmente `False` para todos) es el control de avance que el scraper irá actualizando registro a registro, lo que permite pausar y reanudar el proceso de scraping sin perder el trabajo ya hecho.

In [ ]:
# Agregar columnas RUP vacías (las llenará el scraper)
rup_cols = [
    'rup_estado',           # estado_matricula
    'rup_tamano',           # codigo_tamano_empresa
    'rup_empleados',        # numero_empleados
    'rup_municipio',        # municipio_comercial
    'rup_sanciones',        # count de sanciones
    'rup_multas',           # count de multas
    'rup_inhabilidad',      # inhabilidad
    'rup_ano_financiero',   # ano_informacion_financiera
    'rup_activo_total',     # activo_total
    'rup_patrimonio',       # patrimonio_neto
    'rup_ingresos',         # ingresos_actividad_ordinaria
    'rup_utilidad_neta',    # resultado_del_periodo
    'rup_idx_liquidez',     # AC / PC
    'rup_idx_endeudamiento',# PT / AT
    'rup_idx_cobertura',    # UO / GI
    'rup_rent_patrimonio',  # UN / PN
    'rup_rent_activo',      # UN / AT
]

for col in rup_cols:
    df_unificado[col] = np.nan

# Columnas de control
df_unificado['procesado'] = False

# procesar = False para consorcios/UT y NITs todo ceros (no tienen RUP)
nombre_upper = df_unificado['nombre_proveedor'].str.upper().fillna('')
todo_ceros = df_unificado['nit'].str.match(r'^0+$')
consorcios = nombre_upper.str.contains(r'CONSORCIO|UNION TEMPORAL|UNIoN TEMPORAL|U\.T\.|UT ', regex=True)
df_unificado['procesar'] = ~(consorcios | todo_ceros)

# Ordenar
df_unificado = df_unificado.sort_values('nit').reset_index(drop=True)

print(f"Columnas finales ({len(df_unificado.columns)}):")
for c in df_unificado.columns:
    print(f"  {c}")

print(f"\nShape: {df_unificado.shape}")
print(f"procesar=True:  {df_unificado['procesar'].sum():,}")
print(f"procesar=False: {(~df_unificado['procesar']).sum():,}")
df_unificado.head()

## 9. Guardar

Se cierra el notebook persistiendo dos tablas con propósitos distintos, ambas en formato parquet dentro de `DATA_DIR`:

1. **`proponentes_por_proceso_obra.parquet`**: la descarga cruda de la sección 4, sin deduplicar ni transformar (276,770 registros, 9 columnas). Se conserva a este nivel de detalle porque es la fuente que permitirá, en el EDA extendido, recalcular el número de proponentes reales por proceso y cruzarlo contra el identificador de portafolio de cada contrato — no conviene perder el detalle fila a fila descargando solo el agregado.
2. **`proveedores_rup.parquet`**: la lista unificada y depurada de NITs (sección 8), con las columnas RUP vacías y la bandera `procesar` ya calculada. Esta es la tabla de entrada del scraper automático del RUP, que se ejecuta en un proceso aparte (fuera de este notebook) y que irá marcando `procesado=True` a medida que complete cada NIT.

Con esto termina la extracción y estructuración de proponentes: el resultado queda listo para dos consumidores aguas abajo distintos — el EDA extendido (que necesita el conteo de proponentes por proceso) y el scraper del RUP (que necesita la lista depurada de NITs por consultar).

In [ ]:
# Parquet crudo de proponentes (descarga original)
output_raw = DATA_DIR + 'proponentes_por_proceso_obra.parquet'
df.to_parquet(output_raw, index=False)
print(f"Guardado: {output_raw}")
print(f"  {len(df):,} registros, {df.shape[1]} columnas")

# Dataset unificado para scraper RUP
output_rup = DATA_DIR + 'proveedores_rup.parquet'
df_unificado.to_parquet(output_rup, index=False)
print(f"\nGuardado: {output_rup}")
print(f"  {len(df_unificado):,} NITs únicos, {df_unificado.shape[1]} columnas")
print(f"  Pendientes de procesar: {(~df_unificado['procesado']).sum():,}")

## Anexo — Explicación de este notebook en lenguaje no técnico

*Fuente: `trabajo-de-grado-final/notebooks-explicados/01-extraccion-y-estructuracion-compartida/06_proponentes_por_proceso.md`, preparado para el Anexo A del trabajo de grado.*

### Explicación técnica paso a paso

1. **Por qué hace falta un notebook aparte.** Los datos de "número de oferentes" que ya aparecen en la fuente de procesos (notebook 02) son conteos agregados que el propio SECOP calcula, pero no siempre coinciden con el detalle real de quién se presentó. Para verificar y enriquecer esa información, se descarga por separado el listado detallado de proponentes (empresas que efectivamente radicaron oferta) por cada proceso de contratación.

2. **Descarga por lotes.** Al igual que en el notebook 03, esta fuente se descarga filtrando por los identificadores de proceso ya conocidos (los asociados a los 48.331 contratos de obra pública), en lugar de descargar el listado completo de proponentes de todos los procesos de SECOP (que incluiría procesos de servicios, compras, consultorías, etc., irrelevantes para este trabajo). Se obtienen 276.770 registros de proponentes.

3. **Conteo por proceso.** Se agrupa la tabla de proponentes por identificador de proceso y se cuenta cuántas empresas distintas aparecen en cada uno, generando así un número de oferentes reales por proceso, calculado directamente desde el detalle y no desde un agregado previo del sistema.

4. **Persistencia.** Esta tabla de conteo de proponentes por proceso se guarda para ser cruzada más adelante, en el notebook 01 del EDA extendido, contra el identificador de portafolio de cada contrato — permitiendo así construir la variable de "competencia real" que se usa para poner a prueba la hipótesis H0.

### Conclusión

**Qué se hizo, en palabras simples**

Se descargó, para cada proceso de contratación, la lista de empresas que se presentaron a competir (no solo la que ganó). Esto permite calcular, para cada contrato, cuántos oferentes reales hubo — un dato distinto y más preciso que solo mirar la modalidad de contratación.

**Por qué importa**

Este dato terminó siendo clave para poner a prueba, con evidencia directa, si la competencia entre empresas realmente influye en que un contrato se demore más o menos — uno de los hallazgos más importantes de este trabajo (ver sección 6.3.1 del documento principal).